# Modelo de Recomendación de Alcance — 2. Entrenamiento y evaluación

Entrena el clasificador que estima la probabilidad de que una **suspensión salga efectiva** según
el alcance (canasta) y el historial del NIC, con el dataset de `1_CreacionVariables.ipynb`.

**Requisito:** haber corrido el notebook 1 (`datos/df_universo_alcance.parquet` y
`datos/features_alcance.json`).

## Qué importa medir aquí

La matriz de la fase 3 va a **comparar probabilidades entre canastas** para el mismo NIC. Por eso
no basta con que el modelo ordene bien (AUC): tiene que estar **calibrado dentro de cada canasta**.
Si sobreestima la Liviana y subestima la Pesada, recomendaría mal aunque su AUC sea bueno.

## 1. Carga

In [ ]:
import json
import os

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 250)
SEED = 42

DIR_DATOS = 'datos'
DIR_MODELOS = 'modelos'
os.makedirs(DIR_MODELOS, exist_ok=True)

df = pd.read_parquet(f'{DIR_DATOS}/df_universo_alcance.parquet')
with open(f'{DIR_DATOS}/features_alcance.json', encoding='utf-8') as f:
    meta = json.load(f)

FEATURES = meta['features']
CATEGORICAS = meta['categoricas']
TARGET = meta['target']
assert not set(FEATURES) & set(meta['prohibidas']), 'Hay una variable prohibida entre las features'

print(f'{len(df):,} suspensiones | {len(FEATURES)} variables | tasa efectiva {df[TARGET].mean():.1%}')

## 2. División por tiempo

Igual que en riesgo social: **por fecha, no al azar**. Con variables acumuladas, mezclar meses deja
que el modelo aprenda del futuro y el resultado sale inflado. Se predice siempre hacia adelante.

| Bloque | Periodo | Para qué |
|---|---|---|
| Train | hasta jul-2026 | Aprender |
| Validación | ago-2026 | Decidir cuándo parar (*early stopping*) |
| Test | sep-2026 en adelante | Medir. Se mira una sola vez |

In [ ]:
TRAIN_FIN = '2026-07'
VALID_MES = '2026-08'

tr = df['PERIODO'] <= TRAIN_FIN
va = df['PERIODO'] == VALID_MES
te = df['PERIODO'] > VALID_MES

X = df[FEATURES].copy()
for c in CATEGORICAS:
    X[c] = X[c].astype(str)
y = df[TARGET]

pd.DataFrame({
    'bloque': ['train', 'validación', 'test'],
    'suspensiones': [tr.sum(), va.sum(), te.sum()],
    'tasa_%': [round(y[m].mean() * 100, 1) for m in (tr, va, te)],
})

## 3. Entrenamiento

Sin `class_weights`: la clase está balanceada (~50 %) y en riesgo social pesar las clases arruinó la
calibración, que aquí es lo que más importa.

In [ ]:
cat_idx = [X.columns.get_loc(c) for c in CATEGORICAS]
pool_train = Pool(X[tr], y[tr], cat_features=cat_idx)
pool_valid = Pool(X[va], y[va], cat_features=cat_idx)
pool_test = Pool(X[te], y[te], cat_features=cat_idx)

PARAMS = dict(
    depth=6,
    learning_rate=0.05,
    l2_leaf_reg=3,
    loss_function='Logloss',
    eval_metric='Logloss',   # Logloss y no AUC: castiga la mala calibración, no solo el mal orden
    # Sin combinaciones automáticas de categóricas: con barrio (miles de valores) son lo que más
    # memoria consume, y en un equipo de 16 GB tumbaban el entrenamiento ("bad allocation"). El cruce
    # que importa, barrio x canasta, ya entra explícito como TASA_EF_BARRIO_CANASTA.
    max_ctr_complexity=1,
    # Regla del negocio: a más propensión de pago, la probabilidad de suspender solo puede bajar.
    # Sin esto el modelo podría dar un % alto a un cliente que el servicio dice que va a pagar.
    monotone_constraints={c: -1 for c in ['PROB_PAGO_SIN_INTERVENCION', 'PROB_PAGO_CON_INTERVENCION']},
    random_seed=SEED,
    train_dir=f'{DIR_MODELOS}/catboost_info',
)

modelo = CatBoostClassifier(iterations=2000, early_stopping_rounds=100, verbose=250, **PARAMS)
modelo.fit(pool_train, eval_set=pool_valid, use_best_model=True)
MEJOR_ITER = modelo.get_best_iteration()
print('mejor iteración:', MEJOR_ITER)

## 4. Evaluación sobre el test

Se compara contra una **línea base**: `TASA_EF_BARRIO_CANASTA`, la efectividad histórica suavizada
del barrio con esa canasta. Es la misma idea de la fórmula fija que se armó antes (tasa del lugar
por alcance), pero a nivel de barrio en vez de zona, así que es una base más exigente. Si el
modelo no le gana, no se justifica.

In [ ]:
p_test = modelo.predict_proba(pool_test)[:, 1]
y_test = y[te].values
p_base = df.loc[te, 'TASA_EF_BARRIO_CANASTA'].values


def metricas(nombre, p):
    return {'modelo': nombre,
            'AUC': round(roc_auc_score(y_test, p), 4),
            'Brier': round(brier_score_loss(y_test, p), 4),
            'LogLoss': round(log_loss(y_test, p), 4),
            'prob_media': round(p.mean(), 4)}


METRICAS = pd.DataFrame([metricas('CatBoost', p_test),
                         metricas('Base: barrio x canasta', p_base),
                         metricas('Base: tasa constante', np.full_like(p_test, y[tr | va].mean()))])
print(f'tasa real en test: {y_test.mean():.4f}')
METRICAS

### 4.1 Calibración global (por decil de probabilidad)

En cada decil, la probabilidad media predicha debería parecerse a la tasa real.

In [ ]:
ev = pd.DataFrame({'real': y_test, 'prob': p_test, 'canasta': df.loc[te, 'CANASTA'].values})
ev['decil'] = pd.qcut(ev['prob'], 10, labels=False, duplicates='drop')
(ev.groupby('decil')
   .agg(suspensiones=('real', 'size'), prob_media=('prob', 'mean'), tasa_real=('real', 'mean'))
   .assign(prob_media=lambda d: (d.prob_media * 100).round(1), tasa_real=lambda d: (d.tasa_real * 100).round(1))
   .sort_index(ascending=False))

### 4.2 Calibración y orden **dentro de cada canasta**

Esta es la tabla que decide si la matriz sirve. Una canasta con `desvío` grande quedaría sobre o
subrecomendada. Las canastas con pocas órdenes en el test tienen cifras poco firmes.

In [ ]:
def por_canasta(g):
    auc = roc_auc_score(g['real'], g['prob']) if g['real'].nunique() == 2 else np.nan
    return pd.Series({'suspensiones': len(g),
                      'prob_media_%': round(g['prob'].mean() * 100, 1),
                      'tasa_real_%': round(g['real'].mean() * 100, 1),
                      'AUC': round(auc, 3)})


def tabla_canasta(prob):
    t = (ev.assign(prob=prob).groupby('canasta')
           .apply(por_canasta, include_groups=False).sort_values('suspensiones', ascending=False))
    t['desvío_pp'] = (t['prob_media_%'] - t['tasa_real_%']).round(1)
    return t


tabla_canasta(p_test)

## 5. El desvío por canasta es una tendencia, no un sesgo fijo

La tabla 4.2 muestra al modelo subestimando la Brigada Pesada (~5 pp) y sobreestimando la Pesada
Disponible (~9 pp). **Se probó recalibrar por canasta y no sirve**, por lo que muestra la tabla de
abajo: el desvío de cada canasta cambia de mes a mes.

- **La Brigada Pesada viene mejorando desde septiembre.** Hasta agosto el modelo la acertaba
  (±2 pp); en octubre ya la subestimaba 7 pp. Un calibrador ajustado con el pasado siempre llega
  tarde: ajustado con agosto no corrigió nada en sep–nov, y con septiembre la dejó en −3 pp a
  cambio de descalibrar las canastas chicas.
- **La Pesada Disponible casi no existía** (9 órdenes al mes) y en octubre saltó a 155. No hay
  historia con qué calibrarla: todos los calibradores probados la empeoraron (+10 a +11 pp).

Lo que sí ayuda es **reentrenar con los meses recientes** (sección 7) y hacerlo **cada mes**. Y en la
matriz, tratar con cautela las canastas con pocas órdenes recientes: es donde el modelo más se
equivoca y donde más fácil recomendaría de más.

In [ ]:
# Desvío (predicho − real, en puntos) por canasta y mes, con el modelo entrenado hasta julio
p_va = modelo.predict_proba(pool_valid)[:, 1]
d = pd.DataFrame({'PERIODO': df.loc[va | te, 'PERIODO'].values,
                  'canasta': df.loc[va | te, 'CANASTA'].values,
                  'real': y[va | te].values,
                  'prob': np.concatenate([p_va, p_test])})  # va va antes que te: mismo orden que df
(d.groupby(['canasta', 'PERIODO'])
   .apply(lambda g: round((g['prob'].mean() - g['real'].mean()) * 100, 1) if len(g) >= 30 else np.nan,
          include_groups=False)
   .unstack())

## 6. Qué pesa en la predicción

Si `CANASTA` y las variables por canasta pesan poco, el modelo casi no distingue entre alcances y la
matriz recomendaría lo mismo para todos: eso se confirma en la fase 3.

In [ ]:
imp = (pd.Series(modelo.get_feature_importance(pool_train), index=FEATURES)
        .sort_values(ascending=False).round(2).rename('importancia'))
imp.head(20).to_frame()

## 7. Modelo final y guardado

Se reentrena con **todos** los datos (train + validación + test) y el número de iteraciones que
encontró el early stopping. Los meses recientes son los que más se parecen a lo que viene, y la
sección 5 muestra que el comportamiento de las canastas está cambiando: dejarlos fuera del modelo de
producción sería quedarse con la foto de julio. Las métricas reportadas son las del test de la
sección 4, que el modelo final ya no permite medir.

**Reentrenar cada mes**, borrando el `.parquet` del notebook 1 para traer los cierres nuevos.

In [ ]:
final = CatBoostClassifier(iterations=MEJOR_ITER + 1, verbose=0, **PARAMS)
final.fit(Pool(X, y, cat_features=cat_idx))

# Comprobación de la regla: subir la propensión de pago no puede subir ninguna probabilidad.
muestra = X.sample(5000, random_state=SEED)
base = final.predict_proba(muestra)[:, 1]
for c in PARAMS['monotone_constraints']:
    subida = final.predict_proba(muestra.assign(**{c: np.minimum(muestra[c] + 0.3, 1)}))[:, 1]
    assert (subida <= base + 1e-9).all(), f'La regla no se cumple con {c}'
print('regla cumplida: más propensión de pago nunca sube el %')
final.save_model(f'{DIR_MODELOS}/Modelo_Alcance.cbm')

with open(f'{DIR_MODELOS}/Metadata_Modelo_Alcance.json', 'w', encoding='utf-8') as f:
    json.dump({
        'features': FEATURES,
        'categoricas': CATEGORICAS,
        'target': TARGET,
        'canastas': sorted(df['CANASTA'].unique().tolist()),
        'iteraciones': MEJOR_ITER + 1,
        'params': {k: v for k, v in PARAMS.items() if k != 'train_dir'},
        'entrenado_hasta': str(df['fecha_cierre'].max().date()),
        'evaluacion': {'validacion': VALID_MES,
                       'metricas_test': METRICAS.to_dict(orient='records')},
    }, f, indent=2, ensure_ascii=False)

print('guardado: modelos/Modelo_Alcance.cbm y Metadata_Modelo_Alcance.json')

## 8. Qué sigue (fase 3)

Para cada NIC de `recomendacion_aire_scr` se arma su fila de variables **una vez por canasta**,
recalculando las que dependen de la canasta (historial del NIC con esa canasta, tasa barrio ×
canasta y municipio × canasta), y gana la de mayor probabilidad. Ahí se revisa que la
recomendación de verdad cambie entre NICs y no salga casi siempre Brigada Pesada.

**Recordatorio:** el alcance no se asigna al azar (chequeo 5 del notebook 1). Una probabilidad alta
dice "a suspensiones parecidas les fue mejor con esta canasta", no que mandarla cause el éxito.